In [ ]:
# Instalar bibliotecas necessárias
!pip install -q pandas matplotlib wordcloud kaggle

##### OBTENÇÃO DE DADOS ATUALIZAODS VIA API BANCO CENTRAL

In [ ]:
import requests
import pandas as pd
from datetime import datetime, timedelta

url = 'https://olinda.bcb.gov.br/olinda/servico/MPV_DadosAbertos/versao/v1/odata/MeiosdePagamentosMensalDA(AnoMes=@AnoMes)?@AnoMes=%27201601%27&$format=json&$select=AnoMes,quantidadePix,valorPix,quantidadeTED,valorTED,quantidadeTEC,valorTEC,quantidadeCheque,valorCheque,quantidadeBoleto,valorBoleto,quantidadeDOC,valorDOC'

response = requests.get(url)

if response.status_code == 200:
    data = response.json()
    payment_methods = pd.DataFrame(data['value'])

	# Translate column names from original in Brazilian Portuguese to English
    column_mapping = {
            "AnoMes": "YearMonth",
            "quantidadePix": "quantityPix",
            "valorPix": "valuePix",
            "quantidadeTED": "quantityTED",
            "valorTED": "valueTED",
            "quantidadeTEC": "quantityTEC",
            "valorTEC": "valueTEC",
            "quantidadeCheque": "quantityBankCheck",
            "valorCheque": "valueBankCheck",
            "quantidadeBoleto": "quantityBrazilianBoletoPayment",
            "valorBoleto": "valueBrazilianBoletoPayment",
            "quantidadeDOC": "quantityDOC",
            "valorDOC": "valueDOC"
    }

    # Rename DataFrame columns
    payment_methods.rename(columns=column_mapping, inplace=True)

    # Save data in CSV file.
    payment_methods.to_csv('brazilian_payment_methods.csv', index=False)

else:
    print(f"Request Error: {response.status_code}")

##### OBTENÇÃO DE DADOS VIA KAGGLE

In [ ]:
# Fazer upload do arquivo kaggle.json
from google.colab import files
files.upload()

# Mover o arquivo para o local correto
!mkdir -p ~/.kaggle
!cp kaggle.json ~/.kaggle/
!chmod 600 ~/.kaggle/kaggle.json

# Baixar os dados do Kaggle
!kaggle datasets download -d clovisdalmolinvieira/brazilian-payment-methods
!unzip brazilian-payment-methods.zip

##### ANÁLISE DE DADOS

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
from wordcloud import WordCloud, ImageColorGenerator
from PIL import Image

In [ ]:
# Carregar dados de métodos de pagamento
dados = pd.read_csv('brazilian_payment_methods.csv')

# Visualizar as primeiras linhas do DataFrame
dados.head()

In [ ]:
# Derretendo o DataFrame para transformar colunas em linhas
df_quantity = pd.melt(dados, id_vars=['YearMonth'],
                      value_vars=['quantityPix', 'quantityTED', 'quantityTEC', 'quantityBankCheck', 'quantityBrazilianBoletoPayment', 'quantityDOC'],
                      var_name='PaymentMethod', value_name='Quantity')

df_value = pd.melt(dados, id_vars=['YearMonth'],
                   value_vars=['valuePix', 'valueTED', 'valueTEC', 'valueBankCheck', 'valueBrazilianBoletoPayment', 'valueDOC'],
                   var_name='PaymentMethod', value_name='Value')

# Substituindo nomes das colunas para apenas o método de pagamento
df_quantity['PaymentMethod'] = df_quantity['PaymentMethod'].str.replace('quantity', '')
df_value['PaymentMethod'] = df_value['PaymentMethod'].str.replace('value', '')

# Combinando os DataFrames de quantidade e valor
df_restruturado = pd.merge(df_quantity, df_value, on=['YearMonth', 'PaymentMethod'])

# Visualizar as primeiras linhas do DataFrame
df_restruturado.head()

In [ ]:
df_restruturado['PaymentMethod'] = df_restruturado['PaymentMethod'].replace({
  'Pix': 'PIX',
  'TED': 'TED',
  'TEC': 'TEC',
  'DOC': 'DOC',
  'BankCheck': 'Cheque',
  'BrazilianBoletoPayment': 'Boleto',
})

In [ ]:
# Analisar a frequência dos métodos de pagamento
metodos_pagamento = df_restruturado.groupby('PaymentMethod')['Quantity'].sum().reset_index()
metodos_pagamento.sort_values(by='Quantity', ascending=False, inplace=True)

# Exibir os métodos de pagamento mais populares
print(metodos_pagamento.head(10))

In [ ]:
# Carregar a imagem da bandeira do Brasil
from google.colab import files
files.upload()
mask = np.array(Image.open('Brasil.png'))
image_colors = ImageColorGenerator(mask)

In [ ]:
# Gerar uma nuvem de palavras para os métodos de pagamento
wordcloud = WordCloud(width=800, height=400, background_color='black', mask=mask).generate(' '.join(metodos_pagamento['PaymentMethod'].unique()))

# Visualizar a nuvem de palavras
plt.figure(figsize=(10, 5))
plt.imshow(wordcloud.recolor(color_func=image_colors), interpolation='bilinear')
plt.axis('off')
plt.show()